# Análisis de Churn (流失预测)

Proyecto para predecir la cancelación de clientes (churn) de una empresa de telecomunicaciones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Carga de Datos

In [ ]:
df = pd.read_csv('data/churn_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
df.info()

In [ ]:
df.describe()

### 2.1 Distribución de la variable objetivo (Churn)

In [ ]:
print("Distribución de Churn:")
print(df['churn'].value_counts())
print("\nPorcentaje:")
print(df['churn'].value_counts(normalize=True) * 100)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df['churn'].value_counts().plot(kind='bar', color=['steelblue', 'coral'], ax=ax)
plt.title('Distribución de Churn', fontsize=14)
plt.xlabel('Churn')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.2 Análisis de Variables Numéricas

In [ ]:
numeric_cols = ['edad', 'meses_contrato', 'factura_mensual', 'GB_consumidos', 'llamadas_soporte', 'cambios_plan']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue='churn', kde=True, ax=axes[i], palette=['steelblue', 'coral'])
    axes[i].set_title(f'Distribución de {col}')

plt.tight_layout()
plt.show()

### 2.3 Correlación entre variables

In [ ]:
df_numeric = df.copy()
df_numeric['churn_encoded'] = (df_numeric['churn'] == 'Si').astype(int)

plt.figure(figsize=(10, 8))
sns.heatmap(df_numeric[numeric_cols + ['churn_encoded']].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlación', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Análisis por variables categóricas

In [ ]:
cat_cols = ['genero', 'estado_civil', 'tipo_plan']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['churn'].apply(lambda x: (x == 'Si').mean() * 100)
    churn_rate.plot(kind='bar', ax=axes[i], color='coral')
    axes[i].set_title(f'Tasa de Churn por {col}')
    axes[i].set_ylabel('Tasa de Churn (%)')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Preparación de Datos

In [ ]:
df_model = df.drop('id_cliente', axis=1)

df_model['churn'] = df_model['churn'].map({'Si': 1, 'No': 0})

label_encoders = {}
for col in ['genero', 'estado_civil', 'tipo_plan']:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

X = df_model.drop('churn', axis=1)
y = df_model['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X.columns.tolist()}")

## 4. Entrenamiento del Modelo

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Modelo entrenado exitosamente!")

## 5. Evaluación del Modelo

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MÉTRICAS DEL MODELO")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC:   {roc_auc:.4f} ({roc_auc*100:.2f}%)")
print("=" * 50)

### Explicación de las Métricas:

- **Accuracy (Precisión global)**: Porcentaje de predicciones correctas sobre el total
- **Precision**: De los predichos como positivos, ¿cuántos realmente lo son?
- **Recall (Sensibilidad)**: De los reales positivos, ¿cuántos detectamos?
- **F1-Score**: Media armónica de Precision y Recall
- **ROC-AUC**: Área bajo la curva ROC (1 = perfecto, 0.5 = aleatorio)

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Matriz de Confusión')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='coral', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

### 5.1 Importancia de Features

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis')
plt.title('Importancia de Features', fontsize=14)
plt.xlabel('Importancia')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nTop features:")
print(feature_importance.head())

## 6. Guardar Modelo y Artefactos

In [ ]:
import os
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/churn_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(label_encoders, 'models/label_encoders.pkl')

print("Modelo y artefactos guardados en la carpeta 'models/'")
print("- churn_model.pkl")
print("- scaler.pkl")  
print("- label_encoders.pkl")